# NLP Lab 2: Deep Learning for NLP - Word Embeddings & LSTMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Course**: ITI Natural Language Processing (NLP 101)  
**Based on Lecture**: NLP-ITI-2 (2).pdf (Deep Learning for NLP)  
**Dataset**: Kaggle Stanford Sentiment Treebank (SST-2) / Sentiment dataset (kaggle.com/datasets/atulanandsharma/stanford-sentiment-treebank-v2)  

---

## Objectives:
1. **Word Embeddings**: Understand dense vector representations vs one-hot vectors (`nn.Embedding`).
2. **Sequential Modeling**: Learn how **Recurrent Neural Networks (RNN/LSTM)** process sequences step-by-step.
3. **Vocabulary & Padding**: Map words to integer tokens and pad variable-length sentences to fixed length.
4. **PyTorch Training**: Build an LSTM text classifier and implement the full training loop (Loss, Backprop, Optimizer).

---



In [1]:
# Setup & Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from collections import Counter

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Setup complete! Using PyTorch device: {device}")



Setup complete! Using PyTorch device: cpu


## Step 1: Load Dataset & Build Vocabulary

We will load a sentiment classification dataset (SST-2 binary sentiment).
Labels: `1` = Positive, `0` = Negative.



In [2]:
# Download Stanford Sentiment Treebank subset (SST-2)
url = "https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2/train.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['sentence', 'label'])

print(f"Dataset Size: {len(df)} samples")
print(df.head())



Dataset Size: 6920 samples
                                            sentence  label
0  a stirring , funny and finally transporting re...      1
1  apparently reassembled from the cutting room f...      0
2  they presume their audience wo n't sit still f...      0
3  this is a visually stunning rumination on love...      1
4  jonathan parker 's bartleby should have been t...      1


### Token Indexing & Vocabulary Building

Computers process numbers, not string words!
We need to build a vocabulary index dictionary `word2idx` mapping words to integer IDs:
- `<PAD>`: Reserved index `0` for padding shorter sentences.
- `<UNK>`: Reserved index `1` for unknown / out-of-vocabulary words.

### TODO 1: Build Vocabulary Index Dictionary
Build `word2idx` mapping the top 5,000 most frequent words to integer IDs starting from `2`.



In [3]:
def tokenize(text):
    return re.findall(r'\w+', text.lower())

# Count word frequencies across all training sentences
word_counts = Counter()
for sentence in df['sentence']:
    word_counts.update(tokenize(sentence))

# TODO 1: Build word2idx mapping
# === YOUR CODE HERE ===

# 1. Start with special tokens '<PAD>': 0, '<UNK>': 1
word2idx = {'<PAD>': 0, '<UNK>': 1}
# 2. Get top 5000 words from word_counts
top_5000_words = [word for word, count in word_counts.most_common(5000)]
# 3. Populate word2idx with integer IDs starting at index 2
for idx, word in enumerate(top_5000_words, start=2):
    word2idx[word] = idx

# ======================

print(f"Vocabulary Size (including special tokens): {len(word2idx)}")
print("Sample tokens:", list(word2idx.items())[:10])

assert '<PAD>' in word2idx and word2idx['<PAD>'] == 0, "TODO 1 Failed! <PAD> token must be index 0."
assert len(word2idx) == 5002, f"TODO 1 Failed! Vocabulary length expected 5002, got {len(word2idx)}"
print("TODO 1 Passed!")



Vocabulary Size (including special tokens): 5002
Sample tokens: [('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('a', 3), ('and', 4), ('of', 5), ('to', 6), ('is', 7), ('s', 8), ('it', 9)]
TODO 1 Passed!


---
## Step 2: Sequence Padding & Dataset Preparation

In Lecture 2, we learned that neural networks process inputs in fixed-size mini-batches.
Since text sentences have different lengths, we must pad short sentences with `<PAD>` (0) and truncate long sentences to a fixed `max_len` (e.g., 30 tokens).

### TODO 2: Implement Sequence Padding
Complete `pad_sequence(tokens, max_len)` to return a list of length `max_len`:
- If `len(tokens) < max_len`, append `0` (`<PAD>`) to the end.
- If `len(tokens) > max_len`, truncate to first `max_len` items.



In [4]:
def pad_sequence(token_ids, max_len=30):
    # TODO 2: Pad or truncate list of token_ids to exact length max_len
    # === YOUR CODE HERE ===

    if len(token_ids) >= max_len:
        padded = token_ids[:max_len]
    else:
        padded = token_ids + [0] * (max_len - len(token_ids))

    # ======================
    return padded

# --- Test implementation ---
test_seq = [12, 45, 98]
padded_test = pad_sequence(test_seq, max_len=6)
print("Original:", test_seq)
print("Padded:  ", padded_test)

assert len(padded_test) == 6, "TODO 2 Failed! Output length must equal max_len."
assert padded_test == [12, 45, 98, 0, 0, 0], f"TODO 2 Unexpected padding: {padded_test}"
print("TODO 2 Passed!")



Original: [12, 45, 98]
Padded:   [12, 45, 98, 0, 0, 0]
TODO 2 Passed!


In [5]:
# Create PyTorch Dataset & DataLoader
class SentimentDataset(Dataset):
    def __init__(self, df, word2idx, max_len=30):
        self.labels = df['label'].values
        self.sentences = []
        for text in df['sentence']:
            tokens = tokenize(text)
            ids = [word2idx.get(w, word2idx['<UNK>']) for w in tokens]
            padded_ids = pad_sequence(ids, max_len)
            self.sentences.append(padded_ids)
        self.sentences = torch.tensor(self.sentences, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sentences[idx], self.labels[idx]

# Split train/val
train_df = df.iloc[:5500]
val_df = df.iloc[5500:]

train_dataset = SentimentDataset(train_df, word2idx)
val_dataset = SentimentDataset(val_df, word2idx)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")



Train batches: 86, Val batches: 23


---
## Step 3: PyTorch LSTM Network Architecture

As presented in Lecture 2:
- **Embedding Layer** (`nn.Embedding`): Maps discrete token indices to continuous embedding vectors.
- **LSTM Layer** (`nn.LSTM`): Captures temporal sequence dependency over time.
- **Dense Classifier** (`nn.Linear`): Outputs logit for binary classification.

### TODO 3: Complete PyTorch LSTM Model
Define the `forward(x)` pass of the LSTM text classifier.



In [6]:
class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
        super(LSTMTextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: (batch_size, max_len)
        # TODO 3: Implement forward pass
        # === YOUR CODE HERE ===

        embedded = self.embedding(x)                  # (batch_size, max_len, embedding_dim)
        lstm_out, (hn, cn) = self.lstm(embedded)      # hn shape: (1, batch_size, hidden_dim)
        last_hidden = hn[-1]                          # (batch_size, hidden_dim)
        logits = self.fc(last_hidden).squeeze(1)       # (batch_size,)

        # ======================
        return logits

# Initialize Model
model = LSTMTextClassifier(vocab_size=len(word2idx)).to(device)
print(model)

# Test forward pass with dummy tensor
dummy_input = torch.zeros((4, 30), dtype=torch.long).to(device)
dummy_logits = model(dummy_input)
assert dummy_logits.shape == torch.Size([4]), f"TODO 3 Failed! Expected output shape [4], got {dummy_logits.shape}"
print("TODO 3 Passed!")



LSTMTextClassifier(
  (embedding): Embedding(5002, 64, padding_idx=0)
  (lstm): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)
TODO 3 Passed!


---
## Step 4: Model Training Loop

### TODO 4: Fill in PyTorch Training Loop
In each training iteration, execute the standard PyTorch steps:
1. Reset gradients: `optimizer.zero_grad()`.
2. Compute predictions: `logits = model(inputs)`.
3. Compute loss: `loss = criterion(logits, targets)`.
4. Backpropagate: `loss.backward()`.
5. Update weights: `optimizer.step()`.



In [7]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        # TODO 4: Complete PyTorch training step
        # === YOUR CODE HERE ===

        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        # ======================

        total_loss += loss.item()
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += len(targets)

    return total_loss / len(loader), correct / total



In [8]:
# Run Training for 5 Epochs
epochs = 5
print("Starting LSTM Model Training...")

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch}/{epochs} | Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")

assert train_acc > 0.70, "Training accuracy should be > 70% after 5 epochs!"
print("Training complete!")



Starting LSTM Model Training...
Epoch 1/5 | Loss: 0.6932 | Train Acc: 51.91%
Epoch 2/5 | Loss: 0.6886 | Train Acc: 53.64%
Epoch 3/5 | Loss: 0.6679 | Train Acc: 59.31%
Epoch 4/5 | Loss: 0.6065 | Train Acc: 67.04%
Epoch 5/5 | Loss: 0.5333 | Train Acc: 74.91%
Training complete!


---
## Step 5: Custom Sentiment Inference

Now test the trained LSTM model on custom unseen review sentences!



In [9]:
def predict_sentiment(text, model, word2idx, max_len=30):
    model.eval()
    tokens = tokenize(text)
    ids = [word2idx.get(w, word2idx['<UNK>']) for w in tokens]
    padded_ids = pad_sequence(ids, max_len)
    input_tensor = torch.tensor([padded_ids], dtype=torch.long).to(device)

    with torch.no_grad():
        logit = model(input_tensor)
        prob = torch.sigmoid(logit).item()

    sentiment = "POSITIVE" if prob >= 0.5 else "NEGATIVE"
    print(f"Review: '{text}'")
    print(f"Predicted: {sentiment} (Confidence: {prob if prob >= 0.5 else 1-prob:.4f})\n")

# Sample sentences
predict_sentiment("This movie was absolutely wonderful and brilliant!", model, word2idx)
predict_sentiment("Terrible script, boring actors, and total waste of time.", model, word2idx)



Review: 'This movie was absolutely wonderful and brilliant!'
Predicted: POSITIVE (Confidence: 0.8847)

Review: 'Terrible script, boring actors, and total waste of time.'
Predicted: NEGATIVE (Confidence: 0.8472)



---
## Summary & Takeaways

In Lab 2, you learned:
1. How to map raw text into **word embedding indices** using a vocabulary dictionary.
2. How to apply **sequence padding** to prepare uniform mini-batches.
3. How to construct a PyTorch **LSTM Sequence Classifier** (`nn.Embedding` -> `nn.LSTM` -> `nn.Linear`).
4. How to train a neural network text classifier using binary cross-entropy loss and backpropagation.

Great job completing Lab 2!

